In [13]:
"""
tratar_csv.py - Trata os CSVs de uma pasta para importar no Supabase/PostgreSQL.
"""
import csv
import re
import sys
import unicodedata
from pathlib import Path

import pandas as pd

# ----------------- CONFIGURAÇÃO -----------------
PASTA_ENTRADA = Path(r"C:\Users\FERNANDO\Documents\Projetos\Mercadinho_ERP\base_csv\original")
PASTA_SAIDA = Path(r"C:\Users\FERNANDO\Documents\Projetos\Mercadinho_ERP\base_csv\tratado")
ARQUIVO = "vendas.csv"  # Deixe "" para tratar todos da pasta
LIMITE_DETECCAO = 0.8  # % mínimo de valores que precisam "parecer" data/número
# --------------------------------------------------------------------


def sem_acento(texto: str) -> str:
    """Tira acentos: 'FÍSICO' -> 'FISICO'."""
    nfkd = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in nfkd if not unicodedata.combining(c))


def limpar_cabecalho(nome: str) -> str:
    """minúsculo + sem acento + só letras/números/_ """
    nome = sem_acento(str(nome)).lower().strip()
    nome = re.sub(r"[^a-z0-9]+", "_", nome)
    return nome.strip("_") or "coluna"


def cabecalhos_unicos(nomes: list[str]) -> list[str]:
    """Se o nome repetir, vira ano, ano_2, ano_3..."""
    vistos: dict[str, int] = {}
    saida = []
    for n in nomes:
        vistos[n] = vistos.get(n, 0) + 1
        saida.append(n if vistos[n] == 1 else f"{n}_{vistos[n]}")
    return saida


def ler_csv(caminho: Path) -> tuple[pd.DataFrame, str]:
    """Descobre sozinho a codificação e o separador, retornando o df e o separador."""
    for enc in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            with open(caminho, "r", encoding=enc) as f:
                amostra = f.read(2048)
                try:
                    dialecto = csv.Sniffer().sniff(amostra, delimiters=";,\\t|")
                    separador = dialecto.delimiter
                except Exception:
                    separador = ";"  # Padrão para o Excel do Brasil

            df = pd.read_csv(
                caminho, sep=separador, engine="python", encoding=enc,
                dtype=str, keep_default_na=False,
            )
            return df, separador
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Não consegui ler {caminho.name}")


# Padrões que "parecem" data / número / moeda (com ou sem R$)
RE_DATA_BR = re.compile(r"^\d{1,2}/\d{1,2}/\d{4}(\s+\d{1,2}:\d{2}(:\d{2})?)?$")
RE_DATA_ISO = re.compile(r"^\d{4}-\d{2}-\d{2}(\s+\d{1,2}:\d{2}(:\d{2})?)?$")
RE_NUM_VIRGULA = re.compile(r"^\s*R?\$?\s*-?(\d{1,3}(\.\d{3})*|\d+),\d+\s*$|^-?\d+([.,]\d+)?$")


def proporcao(serie: pd.Series, padrao: re.Pattern) -> float:
    cheios = serie[serie.str.strip() != ""]
    if cheios.empty:
        return 0.0
    return cheios.str.strip().str.match(padrao).mean()


def converter_data(serie: pd.Series) -> pd.Series:
    s = serie.str.strip()
    vazio = s == ""
    dt = pd.to_datetime(s.where(~vazio), dayfirst=True, errors="coerce")
    tem_hora = (dt.dt.hour.fillna(0) + dt.dt.minute.fillna(0) + dt.dt.second.fillna(0)) > 0
    fmt = "%Y-%m-%d %H:%M:%S" if tem_hora.any() else "%Y-%m-%d"
    out = dt.dt.strftime(fmt)
    return out.where(dt.notna(), serie).where(~vazio, "")


def converter_numero(serie: pd.Series) -> pd.Series:
    """Remove 'R$', espaços, remove pontos de milhar e troca vírgula decimal por ponto."""
    s = serie.astype(str).str.replace("R$", "", regex=False)
    s = s.str.replace(r"\s+", "", regex=True)       # Remove espaços em branco
    s = s.str.replace(".", "", regex=False)        # Remove pontos (milhares)
    s = s.str.replace(",", ".", regex=False)       # Troca vírgula por ponto decimal
    return s


def tratar(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    log = []

    # 1) cabeçalhos
    antigos = list(df.columns)
    df.columns = cabecalhos_unicos([limpar_cabecalho(c) for c in antigos])
    if antigos != list(df.columns):
        log.append("cabeçalhos padronizados")

    # Remover colunas geradas automaticamente pelo banco para evitar conflito no Supabase
    colunas_geradas = ["desconto", "total_a_pagar", "ano"]
    for col_gen in colunas_geradas:
        if col_gen in df.columns:
            df = df.drop(columns=[col_gen])
            log.append(f"coluna gerada removida (calculada pelo banco): {col_gen}")

    for col in df.columns:
        # 3) datas
        if proporcao(df[col], RE_DATA_BR) >= LIMITE_DETECCAO or \
           proporcao(df[col], RE_DATA_ISO) >= LIMITE_DETECCAO:
            df[col] = converter_data(df[col])
            log.append(f"data -> AAAA-MM-DD: {col}")
            continue

        # 4) números / moedas com vírgula ou R$
        if proporcao(df[col], RE_NUM_VIRGULA) >= LIMITE_DETECCAO:
            df[col] = converter_numero(df[col])
            log.append(f"número/moeda formatado para decimal: {col}")
            continue

        # 2) acentos no texto
        df[col] = df[col].map(sem_acento)

    return df, log


def main():
    PASTA_SAIDA.mkdir(exist_ok=True, parents=True)
    
    if ARQUIVO:
        arquivos = [PASTA_ENTRADA / ARQUIVO]
        if not arquivos[0].exists():
            sys.exit(f"Arquivo '{ARQUIVO}' não encontrado em '{PASTA_ENTRADA}'")
    else:
        arquivos = sorted(PASTA_ENTRADA.glob("*.csv"))
        if not arquivos:
            sys.exit(f"Nenhum CSV encontrado em '{PASTA_ENTRADA}'")

    for arq in arquivos:
        df, separador_lido = ler_csv(arq)
        df, log = tratar(df)
        destino = PASTA_SAIDA / arq.name
        
        # Salvando com ';' (utf-8-sig) para abrir perfeito nas colunas do Excel brasileiro
        df.to_csv(destino, index=False, encoding="utf-8-sig", sep=";")
        
        print(f"\n{arq.name}: {len(df)} linhas -> {destino} (Separador utilizado: ';')")
        for item in log:
            print("   -", item)


if __name__ == "__main__":
    main()


vendas.csv: 12698 linhas -> C:\Users\FERNANDO\Documents\Projetos\Mercadinho_ERP\base_csv\tratado\vendas.csv (Separador utilizado: ';')
   - cabeçalhos padronizados
   - coluna gerada removida (calculada pelo banco): desconto
   - coluna gerada removida (calculada pelo banco): total_a_pagar
   - coluna gerada removida (calculada pelo banco): ano
   - número/moeda formatado para decimal: quantidade
   - número/moeda formatado para decimal: parcial
   - data -> AAAA-MM-DD: registro
